In [1]:
# Processing monthly PM2.5 and save as annual mean

In [2]:
import os
import xarray as xr
from utils.utils import fix_months

In [3]:
# === Path Builder ===
def get_file_paths(scenario, ens_num):
    num = f"{ens_num:02d}"
    files = []

    if scenario == "ARISE":
        end = "206912" if ens_num not in [8, 9] else "207012"
        path = os.path.join(
            "/glade/campaign/cesm/collections/ARISE-SAI-1.5/",
            f"b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.0{num}/atm/proc/tseries/month_1/",
            f"b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.0{num}.cam.h0.PM25.203501-{end}.nc"
        )
        files.append(path)

    elif scenario == "SSP245":
        base = os.path.join(
            "/glade/campaign/cesm/collections/CESM2-WACCM-SSP245/",
            f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{num}/atm/proc/tseries/month_1/"
        )
        files.append(os.path.join(base, f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{num}.cam.h0.PM25.201501-206412.nc"))
        end = "210012" if ens_num <= 5 else "206912"
        files.append(os.path.join(base, f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{num}.cam.h0.PM25.206501-{end}.nc"))

    elif scenario == "hist":
        base = os.path.join(
            "/glade/campaign/cesm/development/wawg/WACCM6-TSMLT-HIST/"
            f"b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.0{num}/atm/proc/tseries/month_1/"
        )
        files.append(os.path.join(base, f"b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.0{num}.cam.h0.PM25.197801-199912.nc"))
        files.append(os.path.join(base, f"b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.0{num}.cam.h0.PM25.199912-201412.nc"))

    return files

In [4]:
# === Processing Function ===
def calculate_annual_surface_pm25(pm25_surf):
    # Compute annual mean
    print("Calculating annual mean")
    annual_mean = pm25_surf.groupby("time.year").mean("time")
    return annual_mean

In [5]:
# === Historical ====

In [6]:
# === Path config ===
SAVE_DIR = "/glade/work/awells/air_quality/CESM/pm25/annual_PM25/"
SCENARIOS = ["hist"]
ens_num = 1

# === Main Loop ===
for scenario in SCENARIOS:
    print(f"Processing {scenario}")
    file_list = get_file_paths(scenario, ens_num)
    datasets = []

    for f in file_list:
        if not os.path.exists(f):
            raise ValueError(f"Missing: {f}")

        print(f"Reading {os.path.basename(f)}")
        # Take the surface pressure value
        datasets.append(xr.open_dataset(f)["PM25"][:, -1, :, :])

    # Combine files if multiple
    combined_ds = xr.concat(datasets,
                            dim="time") if len(datasets) > 1 else datasets[0]

    if scenario == "hist":
        try:
            expected_end = "2015-01"
            pm_surf = fix_months(combined_ds, "1978-01",
                                 expected_end, scenario)
            annual_pm25 = calculate_annual_surface_pm25(pm_surf)
        except Exception as e:
            print(f"Error processing ensemble {ens_num:02d}: {e}")
            continue

    # Save output
    if scenario == "hist":
        dates = "1990-2009"

    out_file = f"PM25_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)

    description = ("Annual mean PM2.5 - scripts by A.F. Wells (2025)")
    annual_pm25.attrs = combined_ds.attrs
    annual_pm25.attrs["description"] = description
    annual_pm25.attrs["ensemble_number"] = ens_num
    annual_pm25.attrs["scenario"] = scenario

    print(f"Saving annual PM2.5 to {out_path}")
    annual_pm25.to_netcdf(out_path)

print("Done processing all PM2.5 ensembles.")

Processing hist
Reading b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.001.cam.h0.PM25.197801-199912.nc
Reading b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.001.cam.h0.PM25.199912-201412.nc
Calculating annual mean
Saving annual PM2.5 to /glade/work/awells/air_quality/CESM/pm25/annual_PM25/PM25_CESM2_hist_01_1990-2009.nc
Done processing all PM2.5 ensembles.


In [ ]:
# === Path config ===
SAVE_DIR = "/glade/work/awells/air_quality/CESM/pm25/annual_PM25/"
SCENARIOS = ["ARISE", "SSP245"]


# === Main Loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, ensemble {ens_num:02d}")
        file_list = get_file_paths(scenario, ens_num)
        datasets = []

        for f in file_list:
            if not os.path.exists(f):
                raise ValueError(f"Missing: {f}")

            print(f"Reading {os.path.basename(f)}")
            # Take the surface pressure value
            datasets.append(xr.open_dataset(f)["PM25"][:, -1, :, :])

        # Combine files if multiple
        combined_ds = xr.concat(datasets,
                                dim="time") if len(datasets) > 1 else datasets[0]

        if scenario == "ARISE":
            try:
                expected_end = "2070-12" if ens_num in [8, 9] else "2069-12"
                pm_surf = fix_months(combined_ds, "2035-01",
                                     expected_end, scenario)
                annual_pm25 = calculate_annual_surface_pm25(pm_surf)
            except Exception as e:
                print(f"Error processing ensemble {ens_num:02d}: {e}")
                continue

        if scenario == "SSP245":
            try:
                expected_end = "2100-12" if ens_num <= 5 else "2069-12"
                pm_surf = fix_months(combined_ds, "2015-01",
                                     expected_end, scenario)
                annual_pm25 = calculate_annual_surface_pm25(pm_surf)
            except Exception as e:
                print(f"Error processing ensemble {ens_num:02d}: {e}")
                continue

        # Save output
        if scenario == "ARISE":
            dates = "2035-2069"
        elif scenario == "SSP245":
            dates = "2020-2069"

        out_file = f"PM25_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        description = ("Annual mean PM2.5 - scripts by A.F. Wells (2025)")
        annual_pm25.attrs = combined_ds.attrs
        annual_pm25.attrs["description"] = description
        annual_pm25.attrs["ensemble_number"] = ens_num
        annual_pm25.attrs["scenario"] = scenario

        print(f"Saving annual PM2.5 to {out_path}")
        annual_pm25.to_netcdf(out_path)

print("Done processing all PM2.5 ensembles.")

Processing ARISE, ensemble 01
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.001.cam.h0.PM25.203501-206912.nc
Calculating annual mean
Saving annual PM2.5 to /glade/work/awells/air_quality/CESM/pm25/annual_PM25/PM25_CESM2_ARISE_01_2035-2069.nc
Processing ARISE, ensemble 02
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.002.cam.h0.PM25.203501-206912.nc
Calculating annual mean
Saving annual PM2.5 to /glade/work/awells/air_quality/CESM/pm25/annual_PM25/PM25_CESM2_ARISE_02_2035-2069.nc
Processing ARISE, ensemble 03
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.003.cam.h0.PM25.203501-206912.nc
Calculating annual mean
Saving annual PM2.5 to /glade/work/awells/air_quality/CESM/pm25/annual_PM25/PM25_CESM2_ARISE_03_2035-2069.nc
Processing ARISE, ensemble 04
Reading b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.004.cam.h0.PM25.203501-206912.nc
Calculating annual mean
Saving annual PM2.5 to /glade/work/awells/air_quality/CESM/pm25/annual_PM25/PM25_CESM2_ARISE_04_2035-2069.nc
Processi